# 完整商业方案 —— 企业宣传册生成器

## 练习目标（理念）

把第 1 天的「单页摘要」升级成**复合 LLM 调用（Compound LLM Calls）**产品原型：

- **输入**：公司名称 + 官网 URL
- **输出**：面向潜在客户、投资人与求职者的短宣传册（Markdown）
- **流水线**：抓链接 → LLM 筛相关页 → 抓正文拼上下文 → LLM 写宣传册（可流式）

## 和本课 Day 5 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多次 LLM 调用 | `select_relevant_links` 再 `create_brochure` / `stream_brochure` |
| 结构化 JSON 输出 | `response_format={"type": "json_object"}` |
| One-shot prompting | `link_system_prompt` 里给 JSON 示例 |
| 流式打字机 | `stream=True` + `update_display` |
| 受众定制 | `audience_aware_brochure_system_prompt` |

## 怎么跑

1. 激活 `(llms)` 环境；准备 `.env` 里的 `OPENAI_API_KEY`
2. 确保同目录有 `scraper.py`（提供 `fetch_website_links` / `fetch_website_contents`）
3. 从上到下运行；可把示例 URL 换成你关心的公司官网


In [ ]:
# ========== 导入：环境、展示、抓取、OpenAI ==========

# 导入
# 若导入失败：请确认已激活带 (llms) 的虚拟环境（activated environment）

# 导入标准库 os：读环境变量（如 OPENAI_API_KEY）
import os
# 导入标准库 json：把模型返回的 JSON 字符串解析成 dict
import json
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染与流式刷新
from IPython.display import Markdown, display, update_display
# 从本地 scraper 导入两个抓取函数：链列表 + 页面正文
from scraper import fetch_website_links, fetch_website_contents
# 从 openai 导入 OpenAI 客户端：调用 Chat Completions API
from openai import OpenAI


In [ ]:
# ========== 初始化：加载密钥、校验形态、选定模型 ==========

# Initialize and constants

# 加载 .env；override=True 表示已有环境变量也会被 .env 覆盖
load_dotenv(override=True)
# 从环境读取 OpenAI API Key（不要把密钥写进笔记本正文）
api_key = os.getenv('OPENAI_API_KEY')

# 粗检密钥形态：以 sk-proj- 开头且足够长则打印 OK 文案（文案保持英文）
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    # 可能缺密钥或格式不对：提示去排查笔记本（字符串保持原样）
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# 本流水线默认模型 id（筛链接等步骤用）；字符串必须与账号可用模型一致
MODEL = 'gpt-5-nano'
# 创建默认 OpenAI 客户端（密钥从环境变量自动读取）
openai = OpenAI()


In [ ]:
# ========== 试抓：看一眼主站链出哪些 URL ==========

# 调用 scraper：返回页面上发现的链接列表（可能含相对路径）
links = fetch_website_links("https://edwarddonner.com")
# 直接展示列表，便于下一步理解「为什么要让 LLM 筛选」
links


## 第一步：让 GPT-5-nano 判断哪些链接相关

### 做什么

调用 **gpt-5-nano** 阅读网页链接，并以**结构化 JSON** 回复。  
它应判断相关链接，并把 `/about` 这类相对链接补成 `https://company.com/about`。

### 提示技巧

使用「**单次示例提示（one-shot prompting）**」：在 system prompt 里给出期望 JSON 形状。

这很适合 LLM：需要细腻理解「哪些页对宣传册有用」。若不用 LLM、纯靠解析网页硬编码，会非常难！

### 补充

还有更高级的「**Structured Outputs**」，强制模型按规范回复。第 8 周自主 Agent 项目会讲。


In [ ]:
# ========== link_system_prompt（版本 A）：one-shot JSON 示例 ==========

# ask gpt to get what a useful brochure should include
# 影响行为的英文 prompt 保持原样；后面单元格可能用更新后的同名变量覆盖

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [ ]:
# ========== link_system_prompt（版本 B）：覆盖上一格，措辞略扩展 ==========

# 注意：同名变量再次赋值会覆盖 cell 5；后续 select_relevant_links 用的是这一版
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages etc.
you decide which links are most relevant to include in a brochure about the company.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [ ]:
# ========== 拼 user prompt：站点 URL + 抓到的链接列表 ==========

def get_links_user_prompt(url):
    """把「请筛选相关链接」的说明 + 实际 links 拼成发给模型的 user 文本。"""
    # 英文指令保持原样（影响模型行为）；{url} 填入当前公司站
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    # 现场抓取该站链接列表（依赖 scraper）
    links = fetch_website_links(url)
    # 每行一个链接，追加到 user_prompt 末尾
    user_prompt += "\n".join(links)
    return user_prompt


In [ ]:
# ========== 预览 user prompt：确认链接已被拼进去 ==========

# 打印完整 user 消息，便于肉眼检查 one-shot 流水线的输入长什么样
print(get_links_user_prompt("https://edwarddonner.com"))


In [ ]:
# ========== select_relevant_links（初版）：一次 Chat Completions + JSON ==========

def select_relevant_links(url):
    """调用 MODEL，让模型从全站链接里挑宣传册相关页，返回解析后的 dict。"""
    # chat.completions.create：非流式；response_format 要求返回 json_object
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    # 取出助手消息正文（应是 JSON 字符串）
    result = response.choices[0].message.content
    # 解析为 Python dict，供下游按 links 字段遍历
    links = json.loads(result)
    return links


<!-- 占位：上一格定义函数，下一格试跑 -->


In [ ]:
# ========== 试跑初版：对 edwarddonner.com 筛相关链接 ==========

# 返回形如 {"links": [{"type": "...", "url": "..."}, ...]} 的 dict
select_relevant_links("https://edwarddonner.com")


In [ ]:
# ========== select_relevant_links（带日志版）：覆盖上一版函数 ==========

def select_relevant_links(url):
    """同初版逻辑，额外打印「正在调用哪个模型 / 找到几条链接」。"""
    # 进度日志：便于在复合调用链路里定位卡在哪一步（文案保持英文）
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    # 与初版相同的 Chat Completions + json_object 调用
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    # 取出 JSON 字符串
    result = response.choices[0].message.content
    # 解析为 dict
    links = json.loads(result)
    # 打印命中条数，方便快速目检
    print(f"Found {len(links['links'])} relevant links")
    return links


In [ ]:
# ========== 再试 edwarddonner.com：应看到 Selecting... / Found N... ==========

select_relevant_links("https://edwarddonner.com")


In [ ]:
# ========== 换站试跑：Hugging Face 主站链接筛选 ==========

# 同一函数、不同 URL —— 体会「LLM 筛链」比硬编码规则更通用
select_relevant_links("https://huggingface.co")


## 第二步：生成宣传册！

把落地页 + 相关子页的正文组装进另一条 user 提示，再交给模型写短宣传册。

理念：**复合调用（Compound LLM Calls）**——第一次 LLM 决定「读哪些页」，第二次 LLM 才「写宣传册」。


In [ ]:
# ========== 拼上下文：落地页正文 + 各相关页正文 ==========

def fetch_page_and_all_relevant_links(url):
    """抓主站内容，再按 LLM 选出的链接逐页抓取，拼成一大段 Markdown 上下文。"""
    # 主站（落地页）正文
    contents = fetch_website_contents(url)
    # 第一次 LLM 调用：得到 {"links": [...]} 
    relevant_links = select_relevant_links(url)
    # 用 Markdown 小标题组织上下文，方便第二次 LLM 阅读
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    # 遍历每条相关链接：先写类型标题，再追加该页正文
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result


In [ ]:
# ========== 预览拼好的上下文（会触发筛链 + 多页抓取） ==========

# 打印可能很长；用来检查 Landing Page / Relevant Links 结构是否正确
print(fetch_page_and_all_relevant_links("https://www.legisys.ai"))


In [ ]:
# ========== brochure_system_prompt：定「写宣传册」角色与语气 ==========

# 专业语气版本（默认启用）；影响行为的英文保持原样
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':
# 若想幽默语气：注释掉上面，取消注释下面整段（演示 tone 只需改 system prompt）

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [ ]:
# ========== 拼宣传册 user prompt：公司名 + 截断后的网页上下文 ==========

def get_brochure_user_prompt(company_name, url):
    """组装第二次 LLM 的 user 消息；超长上下文截到 5000 字符以防爆窗。"""
    # 英文指令保持原样；company_name 会进 prompt
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    # 追加：落地页 + 相关页正文（内部会再调一次 select_relevant_links）
    user_prompt += fetch_page_and_all_relevant_links(url)
    # Truncate if more than 5,000 characters：硬截断，优先保住开头信息
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt


In [ ]:
# ========== 预览宣传册 user prompt（可能触发抓取与筛链） ==========

get_brochure_user_prompt("Legisys", "https://www.legisys.ai")


In [ ]:
# ========== create_brochure：非流式生成并 Markdown 展示 ==========

def create_brochure(company_name, url):
    """第二次 LLM 调用：用宣传册 system + user 生成全文，一次 display。"""
    # 注意：此处模型写死为 gpt-4.1-mini（与筛链用的 MODEL 常量可以不同）
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    # 取出完整助手回复
    result = response.choices[0].message.content
    # 在笔记本里渲染 Markdown 宣传册
    display(Markdown(result))


In [ ]:
# ========== 试跑：为 Legisys 生成非流式宣传册 ==========

create_brochure("Legisys", "https://www.legisys.ai")


## 最后——一个小改进

只需小改，就能让结果从 OpenAI **流式（streaming）**返回，  
带上熟悉的打字机动画效果（`stream=True` + `update_display`）。


In [ ]:
# ========== stream_brochure：流式打字机版宣传册 ==========

def stream_brochure(company_name, url):
    """与 create_brochure 同提示，但边收 token 边刷新 Markdown 显示。"""
    # stream=True：返回可迭代的 chunk 流，而不是一次性完整 response
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    # 累积已生成文本
    response = ""
    # 先占一个可更新的显示位（display_id）
    display_handle = display(Markdown(""), display_id=True)
    # 逐 chunk 追加；delta.content 可能为 None（用 or ''）
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 试跑流式：Legisys 宣传册打字机输出 ==========

stream_brochure("Legisys", "https://www.legisys.ai")


## 面向受众的宣传册生成

同一套网页上下文，按 **目标受众（target audience）** 与 **角色（role）** 调整语气、结构与侧重点。  
这是「提示工程控制输出形态」的进阶练习。


In [ ]:
# ========== 受众列表 + 受众感知版 system prompt ==========

# 可选目标受众标签（演示用枚举；真正进 prompt 的是选中的字符串）
target_audience = ["B2B", "B2C", "Enterprise", "SMB", "Technical", "non-technical"]
# 可选角色 / 视角
role = ["Executives", "Legal", "Sales", "Investors", "Customers", "Prospective Employees"]

# 长 system prompt：要求模型按 audience + role 调整 tone/structure/emphasis（英文保持原样）
audience_aware_brochure_system_prompt = """
You are an AI assistant that analyzes the contents of several relevant pages from a company website
and creates a concise, professional brochure.

The user will provide two key inputs:
1. Target audience (e.g., customers, investors, recruits)
2. Role or perspective (e.g., buyer, decision-maker, engineer, executive, job seeker)

You must adapt the brochure’s:
- Tone
- Structure
- Emphasis
- Level of detail

based explicitly on the provided target audience and role.

Your responsibilities:

1. Extract and synthesize brochure-worthy content from the website, including:
   - Company overview and positioning
   - Product or service summary
   - Core value proposition
   - Differentiators and strengths

2. Adjust emphasis by audience:
   - **Customers** → focus on problems solved, benefits, usability, outcomes
   - **Investors** → focus on vision, market, differentiation, traction, leadership
   - **Recruits** → focus on culture, mission, team, growth, and career opportunities

3. Adjust framing by role:
   - Highlight what matters most to that role
   - Avoid unnecessary details irrelevant to that role
   - Use language appropriate for the role’s level of technical or business familiarity

4. Include the following sections **only if relevant and supported by the website content**:
   - Company culture and values
   - Customers or partners
   - Careers or job opportunities

5. Exclude or de-prioritize:
   - Navigation menus and UI elements
   - Legal, policy, or compliance pages
   - Blog posts and technical documentation unless directly relevant
   - Repetitive or low-value marketing language

6. Do NOT invent facts or claims not present on the website.
   - If important brochure elements are missing, omit them quietly rather than guessing.

Output requirements:
- Do NOT use code blocks
- Keep the brochure concise, polished, and audience-appropriate
"""


In [ ]:
# ========== 受众版 user prompt：塞入 audience / role + 截断上下文 ==========

def get_audience_aware_brochure_user_prompt(company_name, url, target_audience, role):
    """拼受众定制 user 消息；同样把网页上下文截到 5000 字符。"""
    # 英文指令保持原样；四个参数都会嵌进 prompt
    user_prompt = f"""
You are analyzing a company called: {company_name}

Target audience: {target_audience}
Role / perspective: {role}

You are provided with the contents of the company’s landing page and other relevant pages.
Using this information, create a concise, professional brochure tailored specifically
to the given target audience and role.

Adapt the tone, structure, and emphasis to what matters most to this audience and role.

Respond in markdown without code blocks.
Do not invent information that is not present in the provided content.
Only include sections (e.g., culture, customers, careers) if supported by the content.

Below is the website content:
"""

    # 追加复合抓取得到的上下文
    user_prompt += fetch_page_and_all_relevant_links(url)
    # Truncate if more than 5,000 characters
    user_prompt = user_prompt[:5_000]  # Truncate if more than 5,000 characters
    return user_prompt


In [ ]:
# ========== stream_audience_aware_brochure：受众定制 + 流式展示 ==========

def stream_audience_aware_brochure(company_name, url, target_audience, role):
    """用受众感知 system/user，流式生成并刷新 Markdown。"""
    # 与 stream_brochure 相同流式模式，只换 system/user 构造函数
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": audience_aware_brochure_system_prompt},
            {"role": "user", "content": get_audience_aware_brochure_user_prompt(company_name, url, target_audience, role)}
          ],
        stream=True
    )    
    # 累积文本
    response = ""
    # 可更新显示句柄
    display_handle = display(Markdown(""), display_id=True)
    # 逐块刷新
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


## 思路：从列表选择受众与角色

用户从我们给出的 `target_audience` / `role` 列表里取值，再据此生成宣传册。  
下一格示例：`target_audience[0]`（B2B）+ `role[1]`（Legal）。


In [ ]:
# ========== 示例调用：B2B × Legal 视角的 Legisys 宣传册 ==========

# target_audience[0] → "B2B"；role[1] → "Legal"；可改下标做 A/B 对比
stream_audience_aware_brochure("Legisys", "https://www.legisys.ai", target_audience[0], role[1])


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">本练习扩展了第 1 天代码：多次调用 LLM 并生成文档。

这或许是 <b>Agentic AI</b> 设计模式的第一个例子——组合多次 LLM 调用。第 2 周会更多；第 8 周构建完全自主 Agent 时会大规模回归。

这种方式生成内容是最常见用例之一。与摘要一样，可应用于任何业务领域：营销文案、从规格生成产品教程、个性化邮件等。探索如何应用到你的业务，并做个概念验证。也看看 community-contributions 里同学们的作品——宝藏很多！</span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">进入第 2 周之前（那周超好玩）</h2>
            <span style="color:#900;">请完成 week1 EXERCISE notebook 作为第 1 周末挑战。这能给你前沿 API 的关键练习，并为第 2 周做好准备。</span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">提醒：3 个实用资源</h2>
            <span style="color:#f71;">1. 课程资源见 <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">这里</a>。<br/>
            2. 我的 LinkedIn 在<a href="https://www.linkedin.com/in/eddonner/">这里</a>，很乐意与学员连接！<br/>
            3. 我在尝试 X/Twitter：<a href="https://x.com/edwarddonner">@edwarddonner</a>，希望大家教我怎么玩……  
            </span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">最后！有一个特别请求</h2>
            <span style="color:#090;">
                编辑告诉我：学员在 Udemy 评分影响巨大——这是 Udemy 决定是否推荐课程的主要方式之一。若你能花一分钟评分，我将非常感激！无论如何，需要帮助随时邮件 ed@edwarddonner.com。
            </span>
        </td>
    </tr>
</table>
